In [1]:
import numpy as np
from scipy.sparse import coo_matrix
rng = np.random.default_rng()

In [19]:
def getRegularH(N, dv, dc):
    """
    get parity-check matrix of a regular [dv,dc] LDPC code using the socket
    and permutation technique
    """
    # number of edges
    E = N*dv

    # number of check nodes
    M = E/dc

    if M != np.floor(M):
        assert False, "Code construction with given size not possible, as it would lead to a non-integer number of check nodes"
    
    # nodes connected to sockets
    idx_j = np.ceil(np.arange(1, E+1) / dc).astype(int) - 1
    idx_i = np.ceil(np.arange(1, E+1) / dv).astype(int) - 1

    # random permutation
    idx_j = rng.permutation(idx_j)

    # try to construct matrix and eliminate double edges
    abort = False

    while not abort:
        H = coo_matrix((np.ones_like(idx_i), (idx_j, idx_i)), shape=(int(M), N)).toarray()

        if not np.any(H == 2):
            abort = True
        else:
            # eliminate double edges
            for k in range(N):
                ti = np.where(idx_i == k)[0]

                unique_idx, counts = np.unique(idx_j[ti], return_counts=True)
                duplicates = unique_idx[counts > 1] # duplicates
                if duplicates.size > 0:
                    largeval = duplicates[0]
                    largeidx = np.where(idx_j[ti] == largeval)[0][0]
                    temp = idx_j[ti[largeidx]]
                    rv = rng.integers(idx_j.size)
                    idx_j[ti[largeidx]] = idx_j[rv]
                    idx_j[rv] = temp
    
    return H

In [21]:
#H = getRegularH(12, 4, 6)
#print(H)